# Модуль 1 · Урок 6. Pandas: очистка данных
**AI-Driven ML Engineer · Data Science Academy**

| Время | Блок |
|---|---|
| 0–10 | Разминка «что выведет код» |
| 10–25 | Мостик с урока 5: откуда «−22,7%» · устная проверка |
| 25–60 | Текст в числа, города в 14 написаниях · Практика 1 |
| 60–95 | Пропуски и дубли · Практика 2 |
| 95–110 | Даты и итоговая очистка · Практика 3 |
| 110–120 | Итоги, ДЗ 6 |

**Правило занятия:** предскажи → запусти → объясни. **Правило очистки:** после каждого шага проверяем результат числом.

## Разминка: что выведет код? (30 секунд на вопрос, без запуска)
```python
import numpy as np
import pandas as pd
# 1
s = pd.Series([2, np.nan, 4])
print(s.mean(), np.array([2, np.nan, 4]).mean())
# 2
print(" АЛМАТЫ ".strip().lower())
# 3
print("2026-09-30" < "2026-10", "2026-9-30" < "2026-10")
# 4
df = pd.DataFrame({"id": [1, 1, 2], "x": [5, 5, 7]})
df.drop_duplicates()
print(len(df))
# 5
CITY = {"almaty": "Алматы"}
print(CITY.get("Almaty"), CITY.get("almaty"))
```

In [1]:
import numpy as np
import pandas as pd

In [2]:
s = pd.Series([2, np.nan, 4])

In [3]:
s.mean()

np.float64(3.0)

In [4]:
np.array([2, np.nan, 4]).mean()

np.float64(nan)

In [5]:
print(" АЛМАТЫ ".strip().lower())

алматы


In [6]:
"2026-09-30" < "2026-10"

True

In [7]:
"2026-9-30" < "2026-10"

False

In [8]:
df = pd.DataFrame({"id": [1, 1, 2], "x": [5, 5, 7]})
df.drop_duplicates()


,id,x
0,1,5
2,2,7


In [10]:
df

,id,x
0,1,5
1,1,5
2,2,7


In [9]:
print(len(df))

3


In [12]:
CITY = {"almaty": "Алматы"}
print(CITY.get("Almaty"), CITY.get("almaty"))

None Алматы


## Мостик с урока 5: откуда «−22,7%»
Менеджер сложил выручку оплаченных заказов за сентябрь и октябрь. Повторим его расчёт и уберём TEST. Это Практика 3 урока 5.

In [13]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [108]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))   # с урока 5, сегодня разберём
df["amount"]


0        8070.0
1        3630.0
2        9120.0
3        9350.0
4       11010.0
         ...   
5458    11430.0
5459     3640.0
5460    22150.0
5461    24670.0
5462    14910.0
Name: amount, Length: 5463, dtype: float64

In [17]:
df["order_date"]

0       2026-09-01 08:00
1       2026-09-01 08:00
2       2026-09-01 08:00
3       2026-09-01 08:00
4       2026-09-01 09:00
              ...       
5458    2026-10-31 21:00
5459    2026-10-31 21:00
5460    2026-10-31 22:00
5461    2026-10-31 22:00
5462    2026-10-31 22:00
Name: order_date, Length: 5463, dtype: object

In [16]:
sep = df["order_date"] < "2026-10"
sep

0        True
1        True
2        True
3        True
4        True
        ...  
5458    False
5459    False
5460    False
5461    False
5462    False
Name: order_date, Length: 5463, dtype: bool

In [18]:
octo = ~sep

In [19]:
octo

0       False
1       False
2       False
3       False
4       False
        ...  
5458     True
5459     True
5460     True
5461     True
5462     True
Name: order_date, Length: 5463, dtype: bool

In [20]:
paid = df["status"] == "оплачен"
paid

0        True
1        True
2        True
3        True
4        True
        ...  
5458     True
5459     True
5460     True
5461    False
5462     True
Name: status, Length: 5463, dtype: bool

In [22]:
not_test = df["customer_id"] != "TEST"
not_test

0       True
1       True
2       True
3       True
4       True
        ... 
5458    True
5459    True
5460    True
5461    True
5462    True
Name: customer_id, Length: 5463, dtype: bool

In [23]:
paid & sep

0        True
1        True
2        True
3        True
4        True
        ...  
5458    False
5459    False
5460    False
5461    False
5462    False
Length: 5463, dtype: bool

In [24]:
df.loc[paid & sep, "amount"]

0        8070.0
1        3630.0
2        9120.0
3        9350.0
4       11010.0
         ...   
2777     6900.0
2778     6030.0
2779     7290.0
2780    14940.0
2781    29630.0
Name: amount, Length: 2536, dtype: float64

In [25]:
df.loc[paid & sep, "amount"].sum()

57489850.0

In [26]:
# Как считал менеджер
s, o = df.loc[paid & sep, "amount"].sum(), df.loc[paid & octo, "amount"].sum()
print(s, o, f"{o / s - 1:.1%}")

57489850.0 44419510.0 -22.7%


In [27]:
# Без TEST
s, o = df.loc[paid & not_test & sep, "amount"].sum(), df.loc[paid & not_test & octo, "amount"].sum()
print(s, o, f"{o / s - 1:.1%}")

37489870.0 34419520.0 -8.2%


In [28]:
# Что ещё не так? Строк больше, чем разных заказов
len(df), df["order_id"].nunique()

(5463, 5283)

> Осталось −8,2%, а по чистым данным урока 4 было −2,3%. Разница – 180 повторов заказов. Сегодня чистим таблицу целиком: суммы, города, пропуски, дубли, даты. В конце получим −2,3% и найдём, что случилось в Шымкенте.

## 1. Текст в числа: amount
Начинаем с исходного файла. Строковые методы `.str` делают для каждого значения столбца то же, что метод строки на уроке 1 делает для одной строки.

In [29]:
df = pd.read_csv("orders.csv")
df["amount"].head()

0     8070.0
1     3630.0
2     9120.0
3     9350.0
4    11010.0
Name: amount, dtype: object

In [31]:
type(df['amount'][0])

str

In [32]:
# Сколько сумм записано с пробелом
df["amount"].str.contains(" ")

0       False
1       False
2       False
3       False
4       False
        ...  
5458    False
5459    False
5460    False
5461    False
5462    False
Name: amount, Length: 5463, dtype: bool

In [33]:
df["amount"].str.contains(" ").sum()

445

In [35]:
df.loc[df["amount"].str.contains(" "), "amount"].head()

27       6 470
33       5 070
34      19 490
40       7 570
46       8 350
         ...  
5408    16 300
5437     7 300
5438    75 340
5443     5 850
5448    22 580
Name: amount, Length: 445, dtype: object

In [37]:
# ПРЕДСКАЖИТЕ: errors="coerce" превращает то, что не получилось перевести, в NaN
bad = pd.to_numeric(df["amount"], errors="coerce")
bad

0        8070.0
1        3630.0
2        9120.0
3        9350.0
4       11010.0
         ...   
5458    11430.0
5459     3640.0
5460    22150.0
5461    24670.0
5462    14910.0
Name: amount, Length: 5463, dtype: float64

In [43]:
bad.isna()

0       False
1       False
2       False
3       False
4       False
        ...  
5458    False
5459    False
5460    False
5461    False
5462    False
Name: amount, Length: 5463, dtype: bool

In [39]:
bad.isna().sum()

445

> **Частая ошибка в коде от AI:** `pd.to_numeric(..., errors="coerce")` без проверки. Код не падает, а 445 сумм молча стали NaN, и выручка посчитана без них.

In [40]:
df["amount"]

0        8070.0
1        3630.0
2        9120.0
3        9350.0
4       11010.0
         ...   
5458    11430.0
5459     3640.0
5460    22150.0
5461    24670.0
5462    14910.0
Name: amount, Length: 5463, dtype: object

In [41]:
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
df["amount"].dtype

dtype('float64')

In [42]:
# Проверка: пропусков нет, сумма как на уроке 3
df["amount"].isna().sum(), df["amount"].sum()

(0, 109883360.0)

## 2. Текст: города в 14 написаниях

In [44]:
df["city"].value_counts()

city
Алматы       1927
Астана       1226
Шымкент       711
Караганда     538
Актобе        397
Атырау        146
астана         84
Astana         80
Almaty         69
 Алматы        66
алматы         64
АЛМАТЫ         62
шымкент        54
Shymkent       39
Name: count, dtype: int64

In [47]:
df["city"].str.strip()

0          Алматы
1          Астана
2       Караганда
3          Алматы
4          Астана
          ...    
5458       Алматы
5459    Караганда
5460       Астана
5461       Алматы
5462       Алматы
Name: city, Length: 5463, dtype: object

In [48]:
df["city"].str.strip().str.lower()

0          алматы
1          астана
2       караганда
3          алматы
4          астана
          ...    
5458       алматы
5459    караганда
5460       астана
5461       алматы
5462       алматы
Name: city, Length: 5463, dtype: object

In [49]:
# Убираем пробелы по краям и приводим к нижнему регистру
df["city"].str.strip().str.lower().value_counts()

city
алматы       2119
астана       1310
шымкент       765
караганда     538
актобе        397
атырау        146
astana         80
almaty         69
shymkent       39
Name: count, dtype: int64

Осталось 9 значений: латиница `almaty`, `astana`, `shymkent` не совпадает с кириллицей. Сводим их словарём, как в ДЗ 2 и ДЗ 4. `map` подставляет значение из словаря для каждой строки.

In [50]:
CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

In [51]:
pd.Series(["almaty", "алматы", "Almaty"])

0    almaty
1    алматы
2    Almaty
dtype: object

In [52]:
# ПРЕДСКАЖИТЕ: что map вернёт для ключа, которого нет в словаре?
pd.Series(["almaty", "алматы", "Almaty"]).map({"almaty": "Алматы", "алматы": "Алматы"})

0    Алматы
1    Алматы
2       NaN
dtype: object

In [53]:
df["city"].str.strip().str.lower().map(CITY)

0          Алматы
1          Астана
2       Караганда
3          Алматы
4          Астана
          ...    
5458       Алматы
5459    Караганда
5460       Астана
5461       Алматы
5462       Алматы
Name: city, Length: 5463, dtype: object

In [109]:
df["city"] = df["city"].str.strip().str.lower().map(CITY)
df["city"].isna().sum()      # проверка: все написания нашлись в словаре

0

In [55]:
df["city"].value_counts()

city
Алматы       2188
Астана       1390
Шымкент       804
Караганда     538
Актобе        397
Атырау        146
Name: count, dtype: int64

## Практика 1, 10 минут
1. **[База]** Сколько сумм было записано с пробелом? После перевода: какой тип у `amount` и какая сумма всех заказов?
2. **[База]** Сколько заказов в каждом городе после очистки? Сколько заказов «нашлось» у Алматы и Шымкента по сравнению с написанием `Алматы` и `Шымкент` до очистки?
3. **[Стандарт]** Что будет, если сделать только `.str.lower()` без `.str.strip()`? Сколько значений останется и какое лишнее? Используйте исходный столбец: `pd.read_csv("orders.csv")["city"]`.
4. **[Стандарт]** Сколько выручки потеряется, если перевести `amount` через `errors="coerce"` без `replace`? Сравните две суммы.
5. **[Челлендж]** Уберите `"almaty"` из словаря и сделайте `map`. Сколько NaN? Напишите одну строку, которая показывает, какие исходные написания не нашлись в словаре.

In [ ]:
# 1

In [ ]:
# 2

In [ ]:
# 3

In [ ]:
# 4

In [ ]:
# 5

## 3. Пропуски (04-Missing Data)
NaN – пустая клетка. Методы Pandas `mean`, `sum`, `count` пропуски **пропускают** (вопрос 1 разминки), NumPy – нет.

In [56]:
toy = pd.DataFrame({"A": [1, 2, np.nan],
                    "B": [5, np.nan, np.nan],
                    "C": [1, 2, 3]})
toy

,A,B,C
0,1.0,5.0,1
1,2.0,NaN,2
2,NaN,NaN,3


In [58]:
toy.isna()

,A,B,C
0,False,False,False
1,False,True,False
2,True,True,False


In [57]:
toy.isna().sum()      # пропусков в каждом столбце

A    1
B    2
C    0
dtype: int64

In [59]:
toy.dropna()          # убрать строки, где есть хоть один NaN

,A,B,C
0,1.0,5.0,1


In [60]:
toy.dropna(axis=1)    # убрать столбцы с NaN

,C
0,1
1,2
2,3


In [61]:
toy.dropna(thresh=2)  # оставить строки, где не меньше 2 значений

,A,B,C
0,1.0,5.0,1
1,2.0,NaN,2


In [62]:
toy["A"].mean()

1.5

In [63]:
toy

,A,B,C
0,1.0,5.0,1
1,2.0,NaN,2
2,NaN,NaN,3


In [64]:
toy["A"].fillna(toy["A"].mean())   # заполнить средним

0    1.0
1    2.0
2    1.5
Name: A, dtype: float64

In [65]:
toy

,A,B,C
0,1.0,5.0,1
1,2.0,NaN,2
2,NaN,NaN,3


### Пропуски в orders.csv

In [67]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5463 entries, 0 to 5462
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       5463 non-null   int64  
 1   order_date     5463 non-null   object 
 2   customer_id    5463 non-null   object 
 3   city           5463 non-null   object 
 4   category       5463 non-null   object 
 5   amount         5463 non-null   float64
 6   delivery_days  5303 non-null   float64
 7   status         5463 non-null   object 
dtypes: float64(2), int64(1), object(5)
memory usage: 341.6+ KB


In [66]:
df.isna().sum()

order_id           0
order_date         0
customer_id        0
city               0
category           0
amount             0
delivery_days    160
status             0
dtype: int64

In [68]:
# У каких заказов нет срока доставки?
df.loc[df["delivery_days"].isna(), "status"].value_counts()

status
оплачен    146
отменён     14
Name: count, dtype: int64

| Что делаем с NaN | Когда | Чем опасно |
|---|---|---|
| `dropna()` по одному столбцу | Считаем показатель по этому столбцу (скорость доставки) | Ничего не выдумываем, но строк меньше |
| `fillna(медиана)` | Нужен полный столбец, пропусков мало | Мы придумали значения, доли сдвигаются |
| `fillna(0)` | Почти никогда для сроков и сумм | 0 дней – быстрее любой реальной доставки |
| `dropna()` по всей таблице | Почти никогда | Теряем строки с нормальной выручкой |

In [69]:
# ПРЕДСКАЖИТЕ: три ответа на вопрос «доля доставки за 2 дня в Шымкенте»
d = df.loc[df["city"] == "Шымкент", "delivery_days"]
d

5       1.0
19      1.0
25      1.0
30      1.0
31      1.0
       ... 
5409    7.0
5425    7.0
5428    NaN
5441    6.0
5448    5.0
Name: delivery_days, Length: 804, dtype: float64

In [74]:
d <= 2

5        True
19       True
25       True
30       True
31       True
        ...  
5409    False
5425    False
5428    False
5441    False
5448    False
Name: delivery_days, Length: 804, dtype: bool

In [72]:
(d <= 2).sum()

240

In [73]:
len(d <=2)

804

In [75]:
240 / 804

0.29850746268656714

In [71]:
 (d <= 2).mean()

0.29850746268656714

In [76]:
print(f"NaN как 'нет':  {(d <= 2).mean():.1%}")

NaN как 'нет':  29.9%


In [77]:
d.isna().sum()

22

In [78]:
d.dropna()

5       1.0
19      1.0
25      1.0
30      1.0
31      1.0
       ... 
5397    7.0
5409    7.0
5425    7.0
5441    6.0
5448    5.0
Name: delivery_days, Length: 782, dtype: float64

In [80]:
(d.dropna() <= 2).mean()

0.3069053708439898

In [81]:
print(f"dropna:         {(d.dropna() <= 2).mean():.1%}")

dropna:         30.7%


In [82]:
d.median()

5.0

In [83]:
print(f"fillna(медиана): {(d.fillna(d.median()) <= 2).mean():.1%}")

fillna(медиана): 29.9%


## 4. Дубли
`duplicated()` отмечает строку, которая **полностью** повторяет одну из предыдущих. `keep=False` отмечает обе копии.

In [84]:
df.duplicated()

0       False
1       False
2       False
3       False
4       False
        ...  
5458    False
5459    False
5460    False
5461    False
5462    False
Length: 5463, dtype: bool

In [89]:
df.sort_values(by='order_id').head(10)

,order_id,order_date,customer_id,city,category,amount,delivery_days,status
1989,100001,2026-09-22 17:00,C01685,Астана,Красота,3820.0,2.0,оплачен
1428,100002,2026-09-16 14:00,C00270,Астана,Одежда,8100.0,3.0,оплачен
2514,100003,2026-09-28 09:00,C00381,Атырау,Электроника,30830.0,3.0,оплачен
1466,100004,2026-09-16 20:00,C02118,Актобе,Дом,4190.0,3.0,оплачен
1413,100005,2026-09-16 12:00,C01821,Алматы,Одежда,11530.0,3.0,оплачен
1415,100005,2026-09-16 12:00,C01821,Алматы,Одежда,11530.0,3.0,оплачен
1097,100006,2026-09-13 11:00,C00880,Шымкент,Электроника,22000.0,3.0,отменён
2585,100007,2026-09-28 20:00,C01682,Шымкент,Дом,6880.0,1.0,оплачен
1971,100008,2026-09-22 14:00,C02053,Астана,Электроника,24000.0,3.0,оплачен
220,100009,2026-09-03 09:00,C01541,Караганда,Дом,11550.0,3.0,оплачен


In [85]:
df.duplicated().sum()

180

In [93]:
df[df.duplicated(keep=False)].sort_values("order_id").head(6)

,order_id,order_date,customer_id,city,category,amount,delivery_days,status
1415,100005,2026-09-16 12:00,C01821,Алматы,Одежда,11530.0,3.0,оплачен
1413,100005,2026-09-16 12:00,C01821,Алматы,Одежда,11530.0,3.0,оплачен
1791,100027,2026-09-20 12:00,C01590,Алматы,Электроника,42370.0,2.0,оплачен
1792,100027,2026-09-20 12:00,C01590,Алматы,Электроника,42370.0,2.0,оплачен
2273,100090,2026-09-25 17:00,C01719,Алматы,Книги,2820.0,3.0,оплачен
2276,100090,2026-09-25 17:00,C01719,Алматы,Книги,2820.0,3.0,оплачен


In [91]:
# В каком месяце дубли? .str[:7] – первые 7 символов даты, "2026-09"
df.loc[df.duplicated(), "order_date"].str[:7].value_counts()

order_date
2026-09    180
Name: count, dtype: int64

In [96]:
len(df),len(df.drop_duplicates())

(5463, 5283)

In [98]:
df["city"].is_unique

False

In [110]:
df1 = df.drop_duplicates().copy()     # без присваивания таблица не изменится (вопрос 4 разминки)
len(df1), df1["order_id"].is_unique

(5283, True)

> Все 180 дублей в сентябре: они завышали сентябрь и делали «падение» больше. Проверка после шага: `order_id` теперь уникален.

## Практика 2, 20 минут
1. **[База]** Сколько пропусков в каждом столбце? У каких статусов и в каких городах нет `delivery_days`?
2. **[База]** Сколько полных дублей и в каком месяце? Сколько строк после `drop_duplicates`? Проверьте `is_unique`.
3. **[Стандарт]** Какую выручку оплаченных заказов дают дубли? Насколько они завышали сентябрь? (Считайте до `drop_duplicates`: заново прочитайте файл или сохраните дубли в переменную.)
4. **[Стандарт]** Доля доставки за 2 дня в Шымкенте после удаления дублей: `dropna` против `fillna(медиана)`. Какое число честнее показать менеджеру и почему?
5. **[Челлендж]** Сколько строк останется после `df.dropna()` по всей таблице? Сколько выручки оплаченных заказов при этом пропадёт? Почему так нельзя делать перед подсчётом выручки?

In [ ]:
# 1

In [ ]:
# 2

In [ ]:
# 3

In [ ]:
# 4

In [ ]:
# 5

## 5. Даты
Строка `"2026-09-30 21:00"` сравнивается посимвольно (вопрос 3 разминки). `pd.to_datetime` превращает её в дату, и у столбца появляется `.dt`: месяц, день, день недели, час.

In [99]:
x = "2026-09-30 21:00"

In [101]:
x[:4]

'2026'

In [111]:
pd.to_datetime(df1["order_date"])[0]

Timestamp('2026-09-01 08:00:00')

In [112]:
df1["order_date"] = pd.to_datetime(df1["order_date"])

In [114]:
df1["order_date"].dtype

dtype('<M8[ns]')

In [115]:
type(df1["order_date"][0])

pandas._libs.tslibs.timestamps.Timestamp

In [116]:
df1["order_date"].unique()

<DatetimeArray>
['2026-09-01 08:00:00', '2026-09-01 09:00:00', '2026-09-01 10:00:00',
 '2026-09-01 11:00:00', '2026-09-01 12:00:00', '2026-09-01 13:00:00',
 '2026-09-01 14:00:00', '2026-09-01 15:00:00', '2026-09-01 16:00:00',
 '2026-09-01 17:00:00',
 ...
 '2026-10-31 13:00:00', '2026-10-31 14:00:00', '2026-10-31 15:00:00',
 '2026-10-31 16:00:00', '2026-10-31 17:00:00', '2026-10-31 18:00:00',
 '2026-10-31 19:00:00', '2026-10-31 20:00:00', '2026-10-31 21:00:00',
 '2026-10-31 22:00:00']
Length: 908, dtype: datetime64[ns]

In [117]:
df1["order_date"].apply(lambda x: x.month)

0        9
1        9
2        9
3        9
4        9
        ..
5458    10
5459    10
5460    10
5461    10
5462    10
Name: order_date, Length: 5283, dtype: int64

In [118]:
df1["order_date"].dt.month

0        9
1        9
2        9
3        9
4        9
        ..
5458    10
5459    10
5460    10
5461    10
5462    10
Name: order_date, Length: 5283, dtype: int32

In [119]:
df1["order_date"].dt.month.value_counts()

order_date
10    2681
9     2602
Name: count, dtype: int64

In [121]:
df1["order_date"].dt.day_name().head()

0    Tuesday
1    Tuesday
2    Tuesday
3    Tuesday
4    Tuesday
Name: order_date, dtype: object

In [122]:
df1["month"] = df1["order_date"].dt.month
df1 = df1[df1["customer_id"] != "TEST"]     # последний шаг: тестовый клиент
len(df1)

5280

In [ ]:
paid = df1["status"] == "оплачен"
sep = df1["month"] == 9
octo = df1["month"] == 10
sh = df1["city"] == "Шымкент"
paid.sum()

In [127]:
df1['day'] = df1["order_date"].dt.day
df1['hour'] = df1["order_date"].dt.hour

In [130]:
df1['day_name'] = df1["order_date"].dt.day_name()

In [134]:
df1['day of week'] = df1["order_date"].dt.day_of_week

In [135]:
df1

,order_id,order_date,customer_id,city,category,amount,delivery_days,status,month,day,hour,day_name,day of week
0,101674,2026-09-01 08:00:00,C00681,Алматы,Одежда,8070.0,1.0,оплачен,9,1,8,Tuesday,1
1,101214,2026-09-01 08:00:00,C01210,Астана,Красота,3630.0,2.0,оплачен,9,1,8,Tuesday,1
2,102424,2026-09-01 08:00:00,C01408,Караганда,Дом,9120.0,3.0,оплачен,9,1,8,Tuesday,1
3,102333,2026-09-01 08:00:00,C01529,Алматы,Одежда,9350.0,2.0,оплачен,9,1,8,Tuesday,1
4,102564,2026-09-01 09:00:00,C00657,Астана,Дом,11010.0,1.0,оплачен,9,1,9,Tuesday,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5458,104168,2026-10-31 21:00:00,C01057,Алматы,Одежда,11430.0,2.0,оплачен,10,31,21,Saturday,5
5459,104648,2026-10-31 21:00:00,C00629,Караганда,Книги,3640.0,2.0,оплачен,10,31,21,Saturday,5
5460,104615,2026-10-31 22:00:00,C00557,Астана,Электроника,22150.0,NaN,оплачен,10,31,22,Saturday,5
5461,103460,2026-10-31 22:00:00,C02162,Алматы,Электроника,24670.0,2.0,отменён,10,31,22,Saturday,5


## Практика 3: итоговая очистка и Шымкент, 15 минут
1. **[База]** Выручка оплаченных заказов за сентябрь и октябрь по чистым данным и изменение в %. Совпало с уроком 4 (−2,3%)?
2. **[База]** Журнал очистки для менеджера: заполните таблицу в Markdown-ячейке.

| Шаг | Строк | Изменение выручки окт/сент |
|---|---|---|
| Как считал менеджер (только оплаченные) | 5 463 | −22,7% |
| Без дублей | | |
| Без TEST | | |

3. **[Стандарт]** Шымкент: доля доставки за 2 дня (без пропусков) и доля отмен в сентябре и в октябре. Что изменилось?
4. **[Челлендж]** Средний срок доставки в Шымкенте и во всех остальных городах по месяцам. Сформулируйте вывод для менеджера: где и когда искать причину.

In [ ]:
# 1

In [ ]:
# 2

In [ ]:
# 3

In [ ]:
# 4

## Итоги

| Проблема в данных | Инструмент | Проверка после шага |
|---|---|---|
| Суммы-текст `"15 990"` | `.str.replace(" ", "")` + `pd.to_numeric` | тип float64, `isna().sum() == 0` |
| Город в 14 написаниях | `.str.strip().str.lower().map(CITY)` | 6 значений, `isna().sum() == 0` |
| Пропуски | `isna().sum()`, `dropna()` по столбцу, `fillna` | понимаем, что считаем |
| Дубли | `duplicated()`, `df = df.drop_duplicates()` | `order_id.is_unique` |
| Даты-текст | `pd.to_datetime`, `.dt.month`, `.dt.day_name()` | тип datetime64 |

Из «−22,7%» осталось −2,3%, и больше половины этого падения даёт Шымкент, где в октябре изменились сроки доставки. На уроке 7 разложим показатели по городам и месяцам одной строкой `groupby`.

**ДЗ 6:** ноутбук HW06_pandas_cleaning, сдача в папку `hw06` репозитория `dsa-homework`.